In [1]:
# ==========================================
# CELL 1: Imports & Utilities
# ==========================================
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device and create model directory
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('saved_models', exist_ok=True)
print(f"Deploying on: {device}")

# --- Custom Loss Function ---
class OrdinalContrastiveLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, Z, labels):
        Z = F.normalize(Z, p=2, dim=1)
        cosine_sim = F.cosine_similarity(Z.unsqueeze(1), Z.unsqueeze(0), dim=2)
        d_ij = 1 - cosine_sim
        w_ij = torch.abs(labels.unsqueeze(1) - labels.unsqueeze(0)) / 4.0
        attractive_pull = (1 - w_ij) * (d_ij ** 2)
        repulsive_push = w_ij * (torch.clamp(1 - d_ij, min=0) ** 2)
        L_ij = attractive_pull + repulsive_push
        mask = ~torch.eye(Z.size(0), dtype=torch.bool, device=Z.device)
        return L_ij[mask].mean()

# --- Threshold Optimizer ---
class OptimizedRounder:
    def __init__(self):
        self.coef_ = 0
    def _kappa_loss(self, coef, X, y):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]: X_p[i] = 1
            elif pred >= coef[0] and pred < coef[1]: X_p[i] = 2
            elif pred >= coef[1] and pred < coef[2]: X_p[i] = 3
            elif pred >= coef[2] and pred < coef[3]: X_p[i] = 4
            else: X_p[i] = 5
        ll = cohen_kappa_score(y, X_p, weights='quadratic')
        return -ll
    def fit(self, X, y):
        loss_partial = lambda coef: self._kappa_loss(coef, X, y)
        initial_coef = [1.5, 2.5, 3.5, 4.5]
        self.coef_ = minimize(loss_partial, initial_coef, method='nelder-mead')
        return self.coef_['x']
    def predict(self, X, coef):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]: X_p[i] = 1
            elif pred >= coef[0] and pred < coef[1]: X_p[i] = 2
            elif pred >= coef[1] and pred < coef[2]: X_p[i] = 3
            elif pred >= coef[2] and pred < coef[3]: X_p[i] = 4
            else: X_p[i] = 5
        return X_p

C:\Users\ADMIN\Desktop\data_mining_assignment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Deploying on: cpu


In [2]:
# ==========================================
# CELL 2: Data Loading & Datasets
# ==========================================
DATA = '../data/' # Adjust if your folder is different
train_df = pd.read_csv(DATA + 'train.csv')

# Calculate Global Normalizations (Save these for the Test set!)
min_year, max_year = train_df['year'].min(), train_df['year'].max()
venue_map = {venue: idx for idx, venue in enumerate(train_df['venue'].unique())}
num_venues = len(venue_map)

# Apply to Train DF
train_df['year_norm'] = (train_df['year'] - min_year) / (max_year - min_year + 1e-8)
train_df['venue_id'] = train_df['venue'].map(venue_map)

tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

# --- PyTorch Datasets ---
class PaperDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.titles, self.years = df['title'].values, df['year_norm'].values
        self.venues, self.labels = df['venue_id'].values, df['Label'].values

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        encoding = self.tokenizer(str(self.titles[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'year': torch.tensor(self.years[idx], dtype=torch.float),
            'venue': torch.tensor(self.venues[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float)
        }

class InferencePaperDataset(Dataset): # Used for Test Sets (No Labels)
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.titles, self.years, self.venues = df['title'].values, df['year_norm'].values, df['venue_id'].values

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        encoding = self.tokenizer(str(self.titles[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'year': torch.tensor(self.years[idx], dtype=torch.float),
            'venue': torch.tensor(self.venues[idx], dtype=torch.long)
        }

In [3]:
# ==========================================
# CELL 3: Model Definition with Unfreezing
# ==========================================
class OrdinalRatingModel(nn.Module):
    def __init__(self, num_venues, venue_dim=3):
        super().__init__()
        self.scibert = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')
        
        # GRADUAL UNFREEZING: Freeze bottom 10 layers, unfreeze top 2 + pooler
        for name, param in self.scibert.named_parameters():
            if 'encoder.layer.10' in name or 'encoder.layer.11' in name or 'pooler' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False
                
        self.venue_embedding = nn.Embedding(num_venues, venue_dim)
        fusion_dim = 768 + 1 + venue_dim
        
        # Path A: Regression
        self.regressor = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        # Path B: Contrastive
        self.projection_head = nn.Linear(fusion_dim, 64)

    def forward(self, input_ids, attention_mask, year, venue):
        outputs = self.scibert(input_ids=input_ids, attention_mask=attention_mask)
        v1 = outputs.last_hidden_state[:, 0, :] 
        v2 = self.venue_embedding(venue) 
        year = year.unsqueeze(1) 
        V = torch.cat((v1, year, v2), dim=1) 
        return self.regressor(V).squeeze(1), self.projection_head(V)

In [4]:
# ==========================================
# CELL 4: 5-Fold Cross Validation Training
# ==========================================
FOLDS = 5
EPOCHS = 10
ALPHA = 0.1
BATCH_SIZE = 32

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

# Arrays to store Out-Of-Fold (OOF) predictions
oof_predictions = np.zeros(len(train_df))
oof_labels = np.zeros(len(train_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['Label'])):
    print(f"\n{'='*20} FOLD {fold+1}/{FOLDS} {'='*20}")
    
    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    val_data = train_df.iloc[val_idx].reset_index(drop=True)
    
    train_loader = DataLoader(PaperDataset(train_data, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(PaperDataset(val_data, tokenizer), batch_size=BATCH_SIZE, shuffle=False)
    
    model = OrdinalRatingModel(num_venues=num_venues).to(device)
    mse_criterion = nn.MSELoss()
    supcon_criterion = OrdinalContrastiveLoss()
    
    # Differential Learning Rates
    scibert_params = [p for n, p in model.named_parameters() if 'scibert' in n and p.requires_grad]
    other_params = [p for n, p in model.named_parameters() if 'scibert' not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([{'params': scibert_params, 'lr': 2e-5}, {'params': other_params, 'lr': 2e-4}])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    best_val_loss = float('inf')
    best_fold_preds = []
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for batch in pbar:
            optimizer.zero_grad()
            preds, Z = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), 
                             batch['year'].to(device), batch['venue'].to(device))
            labels = batch['label'].to(device)
            
            loss = mse_criterion(preds, labels) + (ALPHA * supcon_criterion(Z, labels))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        model.eval()
        val_loss = 0
        val_preds_fold = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ", leave=False):
                preds, Z = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), 
                                 batch['year'].to(device), batch['venue'].to(device))
                labels = batch['label'].to(device)
                val_loss += (mse_criterion(preds, labels) + (ALPHA * supcon_criterion(Z, labels))).item()
                val_preds_fold.extend(preds.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f'saved_models/best_model_fold_{fold}.pt')
            best_fold_preds = val_preds_fold 
            
    oof_predictions[val_idx] = best_fold_preds
    oof_labels[val_idx] = val_data['Label'].values

print("\nOptimizing Global Thresholds on OOF Predictions...")
optR = OptimizedRounder()
best_global_thresholds = optR.fit(oof_predictions, oof_labels)
final_oof_discrete = optR.predict(oof_predictions, best_global_thresholds)
print(f"Final Thresholds: {best_global_thresholds}")
print(f"Global OOF QWK Score: {cohen_kappa_score(oof_labels, final_oof_discrete, weights='quadratic'):.4f}")


==================== FOLD 1/5 ====================


Loading weights: 100%|████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 54192.09it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical

Epoch 1 | Train Loss: 2.1802 | Val Loss: 1.6586 | LR: 2.00e-05


Epoch 2/10 [Train]: 100%|█████████████████████████████████████████████████████████| 63/63 [14:25<00:00, 13.74s/it, loss=1.6756]
                                                                                                                               

Epoch 2 | Train Loss: 1.5794 | Val Loss: 1.5334 | LR: 2.00e-05


...

Epoch 8 | Train Loss: 1.0341 | Val Loss: 1.1700 | LR: 2.00e-05


Epoch 9/10 [Train]: 100%|█████████████████████████████████████████████████████████| 63/63 [13:11<00:00, 12.57s/it, loss=1.0961]
                                                                                                                               

Epoch 9 | Train Loss: 0.9692 | Val Loss: 1.1091 | LR: 2.00e-05


Epoch 10/10 [Train]: 100%|████████████████████████████████████████████████████████| 63/63 [12:46<00:00, 12.17s/it, loss=0.7723]
                                                                                                                               

Epoch 10 | Train Loss: 0.9242 | Val Loss: 1.0668 | LR: 2.00e-05

Optimizing Global Thresholds on OOF Predictions...
Final Thresholds: [1.77867497 2.64219683 3.16602528 3.89788779]
Global OOF QWK Score: 0.6256


In [5]:
# ==========================================
# CELL 5: Test Inference & Submission
# ==========================================
public_test_df = pd.read_csv(DATA + 'public_test.csv')
private_test_df = pd.read_csv(DATA + 'private_test.csv')

def preprocess_test_data(test_df, min_y, max_y, v_map):
    df = test_df.copy()
    df['year_norm'] = (df['year'] - min_y) / (max_y - min_y + 1e-8)
    df['venue_id'] = df['venue'].map(lambda x: v_map.get(x, 0)) # Unseen venues become 0
    return df

public_processed = preprocess_test_data(public_test_df, min_year, max_year, venue_map)
private_processed = preprocess_test_data(private_test_df, min_year, max_year, venue_map)

public_loader = DataLoader(InferencePaperDataset(public_processed, tokenizer), batch_size=32, shuffle=False)
private_loader = DataLoader(InferencePaperDataset(private_processed, tokenizer), batch_size=32, shuffle=False)

def get_ensemble_predictions(loader, num_folds, device, thresholds):
    ensemble_preds = np.zeros(len(loader.dataset))
    for fold in range(num_folds):
        print(f"  -> Predicting Fold {fold}...")
        fold_model = OrdinalRatingModel(num_venues=num_venues).to(device)
        fold_model.load_state_dict(torch.load(f'saved_models/best_model_fold_{fold}.pt'))
        fold_model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for batch in loader:
                preds, _ = fold_model(batch['input_ids'].to(device), batch['attention_mask'].to(device), 
                                      batch['year'].to(device), batch['venue'].to(device))
                fold_preds.extend(preds.cpu().numpy())
        ensemble_preds += np.array(fold_preds)
        
    averaged_preds = ensemble_preds / num_folds
    return optR.predict(averaged_preds, thresholds).astype(int)

print("Running Public Test...")
public_test_df['Label'] = get_ensemble_predictions(public_loader, FOLDS, device, best_global_thresholds)

print("Running Private Test...")
private_test_df['Label'] = get_ensemble_predictions(private_loader, FOLDS, device, best_global_thresholds)

print("Saving Submission...")
submission_cols = ['id', 'Label']
final_submission = pd.concat([public_test_df[submission_cols], private_test_df[submission_cols]], ignore_index=True)
final_submission.to_csv('submission.csv', index=False)
print(f"Success! submission.csv created with {len(final_submission)} rows.")

Running Public Test...
  -> Predicting Fold 0...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7802.23it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical

  -> Predicting Fold 1...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7308.43it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical

  -> Predicting Fold 2...


...

  -> Predicting Fold 3...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 9950.72it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical

  -> Predicting Fold 4...


Loading weights: 100%|████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 10346.55it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical

Saving Submission...
Success! submission.csv created with 596 rows.
